<a href="https://colab.research.google.com/github/mahajanprachii/credit-card-fraud-detection/blob/main/credit_card_fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn .model_selection import train_test_split
from sklearn .metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [6]:
cr_data=pd.read_csv('/creditcard.csv')

In [7]:
cr_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [8]:
cr_data.isnull().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


In [11]:
cr_data['Class'].value_counts()

,count
Class,
0,284315
1,492


means a very imbalanced data

In [27]:
X = new.drop(columns='Class')
Y = new['Class']

In [28]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    stratify=Y,
    random_state=2
)

In [29]:
from imblearn.over_sampling import SMOTE

In [31]:
smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)

In [32]:
Y_train_smote.value_counts()

,count
Class,
1,394
0,394


In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier



In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    roc_curve
)

In [35]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

In [46]:
M1 = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

M1.fit(X_train_scaled, Y_train_smote)

Y_pred_M1 = M1.predict(X_test_scaled)
Y_prob_M1 = M1.predict_proba(X_test_scaled)[:, 1]

In [47]:
M2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

M2.fit(X_train_smote, Y_train_smote)

Y_pred_M2 = M2.predict(X_test)
Y_prob_M2 = M2.predict_proba(X_test)[:, 1]

In [48]:
M3 = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

M3.fit(X_train_smote, Y_train_smote)
Y_pred_M3 = M3.predict(X_test)
Y_prob_M3 = M3.predict_proba(X_test)[:, 1]

In [49]:
models = {
    "M1 - Logistic Regression": (Y_test, Y_pred_M1, Y_prob_M1),
    "M2 - Random Forest": (Y_test, Y_pred_M2, Y_prob_M2),
    "M3 - XGBoost": (Y_test, Y_pred_M3, Y_prob_M3)
}

for name, (y_true, y_pred, y_prob) in models.items():

    print("\n", "=" * 60)
    print(name)
    print("=" * 60)

    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))
    print("ROC-AUC  :", roc_auc_score(y_true, y_prob))


M1 - Logistic Regression
Accuracy : 0.9390862944162437
Precision: 0.9777777777777777
Recall   : 0.8979591836734694
F1 Score : 0.9361702127659575
ROC-AUC  : 0.9752628324056895

M2 - Random Forest
Accuracy : 0.9289340101522843
Precision: 0.9883720930232558
Recall   : 0.8673469387755102
F1 Score : 0.9239130434782609
ROC-AUC  : 0.9721706864564007

M3 - XGBoost
Accuracy : 0.9289340101522843
Precision: 0.9772727272727273
Recall   : 0.8775510204081632
F1 Score : 0.9247311827956989
ROC-AUC  : 0.9830962688105545
